# Empirical vs synthetic figures

The same two analyses as the synthetic-data figure (`visualization.plot_model_linearity_bias`,
panels a and b), run on real Twitter users with a PHQ-9 score. Needs the outputs of
`sbatch empirical/run_empirical.job` in `~/data/social_twitter/results/`.
Figures are written to `results/figures/` (gitignored).

In [ ]:
import glob
import os
import sys

if os.path.basename(os.getcwd()) == "empirical":
    os.chdir("..")
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from utils.sensitivity.sa_analyze import phq9_adjacent_band_ladder
from utils.visualization import (_band_bias_per_seed, _errorbar_annotated, _MM_BAND_SHORT, _MM_BANDS,
                                 MULTIMODEL_MODEL_MARKERS, MULTIMODEL_PROMPT_COLOURS,
                                 MULTIMODEL_PROMPT_LABELS, MULTIMODEL_SA_PROMPTS,
                                 MULTIMODEL_SA_ROOTS_SEEDED)

RES = os.environ.get("EMPIRICAL_RESULTS", os.path.expanduser("~/data/social_twitter/results"))
FIG = os.path.join(RES, "figures")
os.makedirs(FIG, exist_ok=True)

EMP_COLOUR, EMP_MARKER = "#222222", "s"          # empirical = near-black squares throughout
BASELINE_COLOUR = "#9a9a9a"
STEPS = [f"{a} → {b}" for a, b in zip(_MM_BAND_SHORT[:-1], _MM_BAND_SHORT[1:])]
GEN_STYLE = {"Qwen3.5-27B": "-", "Gemma-4-31B-it": "--"}


def style(ax):
    """The synthetic figures' axis style: dotted y grid, small ticks, some margin."""
    ax.tick_params(axis="y", labelsize=8.5)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    ax.margins(x=0.12, y=0.15)

## 1. Adjacent-band cosine (severity ladder)

**Synthetic:** the same persona generated at every PHQ-9 band (S-BERT, seeded SA runs,
SD over reps). **Empirical:** each user matched to the same-gender, nearest-age user in
the next band (bootstrap 95% CI over pairs), plus a random-partner baseline.
One compares a persona with itself, the other compares matched people, so each gets
its own y-scale: compare the **shape** across steps, not the level.

In [ ]:
synth_ladder = {}
for gen, root in MULTIMODEL_SA_ROOTS_SEEDED.items():
    for prompt, subdir in MULTIMODEL_SA_PROMPTS:
        lad = phq9_adjacent_band_ladder(root, emb_name="embeddings_sbert.npz", subdir=subdir)
        if lad is not None and len(lad) == len(STEPS):
            synth_ladder[(gen, prompt)] = lad
emp_ladder = pd.read_csv(os.path.join(RES, "ladder_matched.csv"))
emp_ladder

In [ ]:
x = np.arange(len(STEPS))
fig, (ax_s, ax_e) = plt.subplots(1, 2, figsize=(7.5, 2.6), gridspec_kw={"wspace": 0.35})

for (gen, prompt), lad in synth_ladder.items():
    _errorbar_annotated(ax_s, lad["cos"].to_numpy(), lad["std"].to_numpy(),
                        MULTIMODEL_PROMPT_COLOURS[prompt], GEN_STYLE.get(gen, "-"),
                        MULTIMODEL_MODEL_MARKERS.get(gen, "o"),
                        f"{gen}, {MULTIMODEL_PROMPT_LABELS[prompt]}", annotate=False)

e = emp_ladder
ax_e.errorbar(x, e["cos"], yerr=[e["cos"] - e["ci_lo"], e["ci_hi"] - e["cos"]], fmt=EMP_MARKER,
              linestyle="-", color=EMP_COLOUR, capsize=2.5, linewidth=1.6, markersize=5.5,
              markeredgecolor="black", markeredgewidth=0.5, zorder=3, label="matched (age, gender)")
ax_e.plot(x, e["cos_random"], linestyle="--", marker="o", color=BASELINE_COLOUR, linewidth=1.4,
          markersize=4.5, zorder=2, label="random partner")

for ax, title in ((ax_s, "(a) Synthetic: same persona"), (ax_e, "(b) Empirical: matched users")):
    ax.set_xticks(x)
    ax.set_title(title, fontsize=9.5)
    ax.set_ylabel("Cosine similarity", fontsize=9.5)
    style(ax)
    ax.legend(fontsize=6.5, framealpha=0.9, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.42))
ax_s.set_xticklabels(STEPS, rotation=30, ha="right", fontsize=8)
ax_e.set_xticklabels([f"{s}\nn={n}" for s, n in zip(STEPS, e["n_pairs"])], rotation=30, ha="right", fontsize=8)

out = os.path.join(FIG, "ladder_synthetic_vs_empirical.png")
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
pd.concat([pd.DataFrame({"source": f"synthetic {g} {p}", "step": STEPS, "cos": l["cos"], "err": l["std"]})
           for (g, p), l in synth_ladder.items()]
          + [pd.DataFrame({"source": "empirical matched", "step": STEPS, "cos": e["cos"],
                           "ci_lo": e["ci_lo"], "ci_hi": e["ci_hi"], "n": e["n_pairs"]}),
             pd.DataFrame({"source": "empirical random", "step": STEPS, "cos": e["cos_random"]})]
          ).to_csv(out.replace(".png", ".csv"), index=False)
print("->", out)

## 2. Assessor bias per band

One panel per MentalBERT+MLP assessor arm, mean ± SD over its 5 seeds.
**Synthetic:** the arm scored on its own held-out synthetic posts (the teacher arm has no
own set in this layout, so it is shown on the human-optimized Qwen posts).
**Empirical:** the same regressors on the Twitter users. Bias = pred − true; above 0
means over-estimation. The y-scale is shared so the arms compare directly.

In [ ]:
ARMS = ["teacher", "qwen27_optimized", "qwen27_minimal", "gemma4_optimized", "gemma4_minimal"]
SYNTH_CORPUS = {"teacher": "qwen27_optimized"}          # every other arm: its own corpus
ARM_COLOUR = {"teacher": "#7f7f7f",
              "qwen27_optimized": MULTIMODEL_PROMPT_COLOURS["human-opt."],
              "gemma4_optimized": MULTIMODEL_PROMPT_COLOURS["human-opt."],
              "qwen27_minimal": MULTIMODEL_PROMPT_COLOURS["minimal"],
              "gemma4_minimal": MULTIMODEL_PROMPT_COLOURS["minimal"]}


def seed_files(d):
    """Per-seed prediction CSVs in an eval dir (the *_summary.csv files are skipped)."""
    return [f for f in sorted(glob.glob(os.path.join(d, "seed*.csv"))) if "summary" not in f]


bias = {}
for arm in ARMS:
    corpus = SYNTH_CORPUS.get(arm, arm)
    bias[arm] = {"synthetic": _band_bias_per_seed(seed_files(f"data/assessors/bert/{arm}/eval/on_{corpus}")),
                 "empirical": _band_bias_per_seed(seed_files(os.path.join(RES, "bert_eval", arm)))}

emp_users = pd.read_csv(seed_files(os.path.join(RES, "bert_eval", ARMS[0]))[0])
n_band = [int(emp_users["true_phq9"].between(lo, hi).sum()) for lo, hi, _ in _MM_BANDS]

In [ ]:
x = np.arange(len(_MM_BAND_SHORT))
fig, axes = plt.subplots(1, len(ARMS), figsize=(11, 2.6), sharey=True, gridspec_kw={"wspace": 0.12})
rows = []
for ax, arm in zip(axes, ARMS):
    for src, colour, marker in (("synthetic", ARM_COLOUR[arm], "o"), ("empirical", EMP_COLOUR, EMP_MARKER)):
        b = bias[arm][src]
        if not len(b):
            continue
        m, s = np.nanmean(b, 0), np.nanstd(b, 0)
        ax.fill_between(x, m - s, m + s, color=colour, alpha=0.2, linewidth=0)
        ax.plot(x, m, "-", marker=marker, color=colour, linewidth=1.6, markersize=5.5,
                markeredgecolor="black", markeredgewidth=0.5, zorder=3)
        rows += [{"arm": arm, "source": src, "band": bl, "bias": mi, "sd": si, "n_seeds": len(b)}
                 for bl, mi, si in zip(_MM_BAND_SHORT, m, s)]
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(arm, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(_MM_BAND_SHORT, rotation=30, ha="right", fontsize=8)
    style(ax)
axes[0].set_ylabel("Bias = mean(pred − true)", fontsize=9.5)

handles = [Line2D([], [], color=c, marker=mk, linewidth=2, markersize=5.5, markeredgecolor="black",
                  markeredgewidth=0.5, label=lab)
           for c, mk, lab in ((MULTIMODEL_PROMPT_COLOURS["human-opt."], "o", "synthetic, human-optimized posts"),
                              (MULTIMODEL_PROMPT_COLOURS["minimal"], "o", "synthetic, minimal-prompt posts"),
                              ("#7f7f7f", "o", "synthetic (teacher arm on human-opt. posts)"),
                              (EMP_COLOUR, EMP_MARKER, "empirical Twitter users"))]
fig.legend(handles=handles, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=7.5,
           framealpha=0.9)
fig.text(0.5, 1.0, "Empirical users per band: " + " · ".join(f"{b} {n}" for b, n in zip(_MM_BAND_SHORT, n_band)),
         ha="center", va="bottom", fontsize=7.5, color="0.3")

out = os.path.join(FIG, "bias_synthetic_vs_empirical.png")
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
pd.DataFrame(rows).to_csv(out.replace(".png", ".csv"), index=False)
print("->", out, "| empirical users per band:", dict(zip(_MM_BAND_SHORT, n_band)))